In [5]:
import autogen
from autogen import ConversableAgent, UserProxyAgent
from typing import Annotated
from IPython.display import display, Markdown

In [2]:
from getpass import getpass
import os 

os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API Key: ")

In [3]:
config_list = {
    "config_list": [{"model": "gpt-4o-mini", "temperature": 0.1}],
}

In [6]:
termination_checker = lambda msg: msg.get("content") is not None and "TERMINATE" in msg["content"]

user_proxy = UserProxyAgent(
    name = "user_bmi_input",
    llm_config = False,
    is_termination_msg=termination_checker,
    human_input_mode="TERMINATE",
    code_execution_config=False
)

bmi_agent = ConversableAgent(
    name = "bmi_calculator",
    system_message = """You're a health expert. You can use the tool to calculate the BMI using the given input
                        and provide the health recommmendation for the user.
                        Return 'TERMINATE' when the task is done.""",
    llm_config = config_list,
    is_termination_msg=termination_checker,
)

diet_agent = ConversableAgent(
    name = "diet_planner",
    system_message = """You're an expert Diet planner given the heath recommendation for the user you have to provides meal suggestions.
                        Return 'TERMINATE' when the task is done.""",
    llm_config = config_list,
    is_termination_msg=termination_checker,
)

workout_agent = ConversableAgent(
    name = "workout_scheduler",
    system_message="""You're a workout scheduler. Given the BMI result, health recommendation, and meal suggestions,
                    create a safe beginner-friendly weekly workout plan.
                    Return 'TERMINATE' when the task is done.""",
    llm_config=config_list,
    is_termination_msg=termination_checker,
)

@user_proxy.register_for_execution()

Note: It does not collect user input.

Its role is: it tells AutoGen which agent is allowed to actually execute the Python function.

So the flow is:
- bmi_agent sees tool schema
- bmi_agent decides: "I need to call bmi_calculator"
- user_proxy executes the actual Python function
- tool result goes back to bmi_agent
- bmi_agent explains/interprets the result

In [7]:
@bmi_agent.register_for_llm(description="Calculate BMI from weight and height, then return the numeric BMI result.")
@user_proxy.register_for_execution()
def bmi_calculator(
    weight_kg: Annotated[float, "User weight in kilograms"],
    height_cm: Annotated[float, "User height in centimeters"],
    age: Annotated[int, "User age in years"],
    gender: Annotated[str, "User gender"],
    dietary_preference: Annotated[str, "Dietary preference such as Veg, Non-Veg, or Vegan"],
) -> dict:
    """Calculate BMI and return a structured BMI result."""
    if weight_kg <= 0:
        raise ValueError("weight_kg must be greater than 0")
    if height_cm <= 0:
        raise ValueError("height_cm must be greater than 0")

    height_m = height_cm / 100
    bmi = round(weight_kg / (height_m ** 2), 2)

    return {
        "age": age,
        "gender": gender,
        "dietary_preference": dietary_preference,
        "weight_kg": weight_kg,
        "height_cm": height_cm,
        "bmi": bmi,
    }

In [8]:
weight_kg = float(input("Enter your weight in kg: "))
height_cm = float(input("Enter your height in cm: "))
age = int(input("Enter your age: "))
gender = input("Enter your gender: ")
dietary_preference = input("Enter dietary preference (Veg/Non-Veg/Vegan): ")

user_profile = {
    "weight_kg": weight_kg,
    "height_cm": height_cm,
    "age": age,
    "gender": gender,
    "dietary_preference": dietary_preference,
}

messages = [
    f"""Use the bmi_calculator tool for this user profile: {user_profile}.
    After the tool returns the result, classify the BMI category and provide a simple health recommendation.""",
    "Create a one-day meal plan using the BMI result, category, health recommendation, and dietary preference from the previous chat.",
    "Create a safe weekly workout schedule using the BMI result, category, health recommendation, and meal suggestions from the previous chats.",
]

In [9]:
results = autogen.initiate_chats(
    [
        {
            "sender": user_proxy,
            "recipient": bmi_agent,
            "message": messages[0],
            "summary_method": "reflection_with_llm",
            "clear_history": True,
        },
        {
            "sender": bmi_agent,
            "recipient": diet_agent,
            "message": messages[1],
            "summary_method": "reflection_with_llm",
        },
        {
            "sender": diet_agent,
            "recipient": workout_agent,
            "message": messages[2],
            "summary_method": "reflection_with_llm",
        }
    ]
)

for result in results:
    display(Markdown(result.summary))


********************************************************************************
Starting a new chat....

********************************************************************************
user_bmi_input (to bmi_calculator):

Use the bmi_calculator tool for this user profile: {'weight_kg': 69.0, 'height_cm': 180.0, 'age': 29, 'gender': 'Male', 'dietary_preference': 'Non-Veg'}.
    After the tool returns the result, classify the BMI category and provide a simple health recommendation.

--------------------------------------------------------------------------------

>>>>>>>> USING AUTO REPLY...
bmi_calculator (to user_bmi_input):

***** Suggested tool call (call_7cNLhe4QGeBb9GRFE7Le77Vc): bmi_calculator *****
Arguments: 
{"weight_kg":69,"height_cm":180,"age":29,"gender":"Male","dietary_preference":"Non-Veg"}
*******************************************************************************

--------------------------------------------------------------------------------

>>>>>>>> USING AUTO

The user's BMI is 21.3, which falls within the Normal weight category. The health recommendation is to maintain a balanced diet and regular physical activity, focusing on lean proteins, whole grains, fruits, and vegetables for optimal nutrition.

The conversation provided a one-day meal plan for a user with a BMI of 21.3, categorized as normal weight. The plan emphasizes a balanced diet with lean proteins, whole grains, fruits, and vegetables, including meals and snacks that promote optimal nutrition while encouraging hydration and portion control based on activity levels.

The conversation provided a weekly workout schedule for someone with a BMI of 21.3, categorized as normal weight. The plan emphasizes a balanced approach to fitness, incorporating cardio, strength training, and flexibility exercises. It encourages maintaining a balanced diet rich in lean proteins, whole grains, fruits, and vegetables, while promoting hydration and active recovery. The schedule includes specific workouts for each day of the week, with a focus on listening to one's body and modifying exercises as needed.